Import + Config + Seed

In [1]:
import math
import random
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List
from collections import deque

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils as nn_utils
import wandb
import copy

@dataclass
class CFG:
    ml100k_path: str = "ml100k_ratings.csv"
    sep: str = ";"

    snapshot_gran: str = "d"
    window_sids: int = 0

    node_dim: int = 64
    embed_dim: int = 64
    compressed_dim: int = 16

    n_layers: int = 2
    dropout: float = 0.1

    lr: float = 3e-4
    weight_decay: float = 1e-5
    epochs: int = 50

    train_ratio: float = 0.70
    val_ratio: float = 0.15
    test_ratio: float = 0.15

    k: int = 20

    seed: int = 1337
    device: str = "cpu"

    debug_sids: int = 3
    full_train_epochs: int = 30

    distillation_weight: float = 0.1
    grad_clip: float = 1.0


cfg = CFG()
wandb.login()

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(cfg.seed)
device = torch.device(cfg.device)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: strannayasonya (strannayasonya-hse-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Load MovieLens-100K → df(from,to,timestamp)

In [2]:
def load_ml100k_as_events(path: str, sep: str = ";") -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=sep,
        header=0,
        names=["user_id", "item_id", "timestamp", "rating"],
        engine="python",
    )
    df = df.dropna(subset=["user_id", "item_id", "timestamp"]).copy()
    df["user_id"] = df["user_id"].astype(str)
    df["item_id"] = df["item_id"].astype(str)
    df["timestamp"] = df["timestamp"].astype("int64")
    df = df.sort_values("timestamp").reset_index(drop=True)

    out = df[["user_id", "item_id", "timestamp"]].rename(
        columns={"user_id": "from", "item_id": "to"}
    )
    return out


df_raw = load_ml100k_as_events(cfg.ml100k_path, cfg.sep)

Bipartite mapping + time starts at 0

In [3]:
def build_bipartite_id_maps(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, int], Dict[str, int]]:
    user_map: Dict[str, int] = {}
    item_map: Dict[str, int] = {}

    def map_user(x: str) -> int:
        if x not in user_map:
            user_map[x] = len(user_map)
        return user_map[x]

    def map_item(x: str) -> int:
        if x not in item_map:
            item_map[x] = len(item_map)
        return item_map[x]

    df = df.copy()
    df["from"] = df["from"].astype(str).map(map_user).astype("int64")
    df["to"] = df["to"].astype(str).map(map_item).astype("int64")

    item_offset = int(len(user_map))
    df["to"] = (df["to"] + item_offset).astype("int64")
    return df, user_map, item_map


df = df_raw.sort_values("timestamp").reset_index(drop=True)
t0 = int(df["timestamp"].min())
df["timestamp"] = (df["timestamp"] - t0).astype("int64")

df, user_map, item_map = build_bipartite_id_maps(df)

num_users = int(len(user_map))
num_items = int(len(item_map))
item_offset = int(num_users)
num_nodes = int(num_users + num_items)

df.head(), (num_users, num_items, num_nodes)

(   from   to  timestamp
 0     0  943          0
 1     0  944         17
 2     0  945         44
 3     0  946         71
 4     0  947        133,
 (943, 1682, 2625))

Temporal split 70-15-15 по событиям

In [4]:
def bounds_event_ratio_split(
    df: pd.DataFrame, train_ratio: float, val_ratio: float, time_col: str = "timestamp"
) -> Tuple[int, int]:
    df_sorted = df.sort_values(time_col).reset_index(drop=True)
    times = df_sorted[time_col].to_numpy()
    n = int(times.shape[0])

    idx_val = int(n * float(train_ratio))
    idx_test = int(n * float(train_ratio + val_ratio))

    val_time = int(times[idx_val])
    test_time = int(times[idx_test])
    return val_time, test_time


val_time, test_time = bounds_event_ratio_split(df, cfg.train_ratio, cfg.val_ratio)
val_time, test_time

(12314561, 16311713)

Discretize timestamps → sid, подготовить события по split и snapshots (undirected

In [5]:
def gran_to_seconds(gran: str) -> int:
    g = gran.lower()
    if g == "h":
        return 3600
    if g == "d":
        return 86400
    raise ValueError("snapshot_gran must be 'h' or 'd'")


bin_sec = gran_to_seconds(cfg.snapshot_gran)

df = df.copy()
df["sid"] = (df["timestamp"] // bin_sec).astype("int64")

df["split"] = np.where(
    df["timestamp"] < val_time,
    "train",
    np.where(df["timestamp"] < test_time, "val", "test"),
)

df_events = df[["from", "to", "timestamp", "sid", "split"]].copy()

df_rev = df_events.copy()
df_rev[["from", "to"]] = df_rev[["to", "from"]]
df_mp = pd.concat([df_events, df_rev], ignore_index=True)
df_mp = df_mp.drop_duplicates(subset=["from", "to", "timestamp"]).sort_values(
    ["sid", "timestamp"]
).reset_index(drop=True)

max_sid = int(df["sid"].max())
max_sid

214

Группировка по sid (events_by_sid + mp_by_sid)

In [6]:
def group_by_sid(df_in: pd.DataFrame, sid_col: str = "sid") -> Dict[int, pd.DataFrame]:
    out: Dict[int, pd.DataFrame] = {}
    for sid, g in df_in.groupby(sid_col, sort=True):
        out[int(sid)] = g.reset_index(drop=True)
    return out

mp_by_sid = group_by_sid(df_mp, "sid")
train_events_by_sid = group_by_sid(df_events[df_events["split"] == "train"], "sid")
val_events_by_sid = group_by_sid(df_events[df_events["split"] == "val"], "sid")
test_events_by_sid = group_by_sid(df_events[df_events["split"] == "test"], "sid")

(len(train_events_by_sid), len(val_events_by_sid), len(test_events_by_sid))

(142, 47, 26)

In [7]:
def select_last_event_per_user(batch_df: pd.DataFrame, item_offset: int) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Внутри одного sid берём ровно 1 позитив на пользователя: последний по timestamp.
    Возвращаем:
      users_global: [U]
      pos_items_local: [U]  (0..num_items-1)
    """
    if len(batch_df) == 0:
        return (
            torch.empty(0, dtype=torch.long),
            torch.empty(0, dtype=torch.long),
        )

    # сортируем (user, timestamp) и берём последний
    g = batch_df.sort_values(["from", "timestamp"], ascending=[True, True])
    # idx последнего события на каждого user
    last_idx = g.groupby("from", sort=False).tail(1).index
    picked = g.loc[last_idx]

    users_global = torch.tensor(picked["from"].to_numpy(dtype=np.int64), dtype=torch.long)
    pos_items_local = torch.tensor((picked["to"].to_numpy(dtype=np.int64) - item_offset), dtype=torch.long)
    return users_global, pos_items_local


Простая GCN-подобная нормализация + Encoder + DotProduct decoder

In [8]:
def mp_edges_for_sid(sid: int, mp_by_sid: Dict[int, pd.DataFrame], device: torch.device) -> Tuple[torch.Tensor, torch.Tensor]:
    g = mp_by_sid.get(int(sid))
    if g is None or len(g) == 0:
        return torch.empty(0, dtype=torch.long, device=device), torch.empty(0, dtype=torch.long, device=device)
    src = torch.tensor(g["from"].to_numpy(np.int64), dtype=torch.long, device=device)
    dst = torch.tensor(g["to"].to_numpy(np.int64), dtype=torch.long, device=device)
    return src, dst


def concat_edges(edge_list: List[Tuple[torch.Tensor, torch.Tensor]]) -> Tuple[torch.Tensor, torch.Tensor]:
    if len(edge_list) == 0:
        return torch.empty(0, dtype=torch.long), torch.empty(0, dtype=torch.long)
    src = torch.cat([s for s, _ in edge_list], dim=0)
    dst = torch.cat([d for _, d in edge_list], dim=0)
    return src, dst


def build_norm_adj(edge_src: torch.Tensor, edge_dst: torch.Tensor, num_nodes: int, device: torch.device) -> torch.Tensor:
    self_idx = torch.arange(num_nodes, dtype=torch.long, device=device)
    src = torch.cat([edge_src, self_idx], dim=0)
    dst = torch.cat([edge_dst, self_idx], dim=0)

    deg = torch.bincount(src, minlength=num_nodes).float()
    deg_inv_sqrt = torch.pow(deg.clamp(min=1.0), -0.5)

    vals = deg_inv_sqrt[src] * deg_inv_sqrt[dst]
    idx = torch.stack([src, dst], dim=0)

    A = torch.sparse_coo_tensor(idx, vals, size=(num_nodes, num_nodes), device=device)
    A = A.coalesce()
    return A

Model

In [9]:
class SimpleGCNEncoder(nn.Module):
    def __init__(self, in_dim: int, hid_dim: int, out_dim: int, n_layers: int, dropout: float):
        super().__init__()
        assert n_layers >= 1
        self.dropout = float(dropout)
        self.lins = nn.ModuleList()
        self.bns = nn.ModuleList()

        if n_layers == 1:
            self.lins.append(nn.Linear(in_dim, out_dim))
        else:
            self.lins.append(nn.Linear(in_dim, hid_dim))
            self.bns.append(nn.BatchNorm1d(hid_dim))
            for _ in range(n_layers - 2):
                self.lins.append(nn.Linear(hid_dim, hid_dim))
                self.bns.append(nn.BatchNorm1d(hid_dim))
            self.lins.append(nn.Linear(hid_dim, out_dim))

    def forward(self, A_norm: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
        h = x
        if len(self.lins) == 1:
            h = torch.sparse.mm(A_norm, self.lins[0](h))
            return h

        for li in range(len(self.lins) - 1):
            h = torch.sparse.mm(A_norm, self.lins[li](h))
            h = self.bns[li](h)
            h = F.relu(h)
            h = F.dropout(h, p=self.dropout, training=self.training)

        h = torch.sparse.mm(A_norm, self.lins[-1](h))
        return h


class Compressor(nn.Module):
    def __init__(self, d_in: int, d_out: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_out * 2),
            nn.ReLU(),
            nn.Linear(d_out * 2, d_out),
            nn.LayerNorm(d_out),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class DotProductDecoder(nn.Module):
    def forward(self, z_users: torch.Tensor, z_items: torch.Tensor) -> torch.Tensor:
        return z_users @ z_items.t()

Prefix-window сборка (по sid) + forward для одного sid

In [10]:
def compute_z_from_prefix(window: deque, num_nodes: int) -> Tuple[torch.Tensor, torch.Tensor]:
    src, dst = concat_edges(list(window))
    A = build_norm_adj(src, dst, num_nodes=num_nodes, device=device)
    z_big = encoder(A, node_emb.weight)
    z_small = compressor(z_big)
    return z_big, z_small

Train epoch: streaming по sid (без лика внутри sid)

In [19]:
def train_epoch_streaming(
    train_events_by_sid: Dict[int, pd.DataFrame],
    mp_by_sid: Dict[int, pd.DataFrame],
    window_sids: int,
    debug: bool = False,
) -> float:
    encoder.train()
    node_emb.train()
    compressor.train()

    item_ids_global = torch.arange(item_offset, item_offset + num_items, device=device, dtype=torch.long)

    window: deque = deque(maxlen=(window_sids if window_sids > 0 else None))
    prefix_sid = 0

    total_loss_big = 0.0
    total_loss_small = 0.0
    total_distill = 0.0
    total_users = 0

    sids_list = sorted(list(train_events_by_sid.keys()))[: cfg.debug_sids] if debug else sorted(train_events_by_sid.keys())

    for sid in sids_list:
        while prefix_sid < sid:
            s, d = mp_edges_for_sid(prefix_sid, mp_by_sid, device)
            if s.numel() > 0:
                window.append((s, d))
            prefix_sid += 1

        z_big, z_small = compute_z_from_prefix(window, num_nodes)

        users_global_cpu, pos_items_local_cpu = select_last_event_per_user(train_events_by_sid[sid], item_offset)
        if users_global_cpu.numel() == 0:
            continue

        users_global = users_global_cpu.to(device)
        pos_items_local = pos_items_local_cpu.to(device)

        users_z_big = z_big[users_global]
        items_z_big = z_big[item_ids_global]
        logits_big = users_z_big @ items_z_big.t()
        loss_big = F.cross_entropy(logits_big, pos_items_local)

        users_z_small = z_small[users_global]
        items_z_small = z_small[item_ids_global]
        logits_small = users_z_small @ items_z_small.t()
        loss_small = F.cross_entropy(logits_small, pos_items_local)

        logits_big_detached = logits_big.detach()
        distill_loss = F.mse_loss(logits_small, logits_big_detached)

        loss = loss_big + loss_small + cfg.distillation_weight * distill_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        nn_utils.clip_grad_norm_(list(encoder.parameters()) + list(compressor.parameters()), cfg.grad_clip)
        opt.step()
        if scheduler is not None:
          scheduler.step()

        U = int(users_global.numel())
        total_loss_big += float(loss_big.item()) * U
        total_loss_small += float(loss_small.item()) * U
        total_distill += float(distill_loss.item()) * U
        total_users += U

    total_loss = float((total_loss_big + total_loss_small + cfg.distillation_weight * total_distill)/max(total_users, 1))
    wandb.log({"train_loss": total_loss})

    log_gradient_stats()
    return total_loss


Eval: streaming по sid

In [12]:
@torch.no_grad()
def eval_streaming(
    events_by_sid: Dict[int, pd.DataFrame],
    mp_by_sid: Dict[int, pd.DataFrame],
    start_prefix_sid: int,
    window_sids: int,
    k: int,
) -> Dict[str, float]:
    encoder.eval()
    node_emb.eval()
    compressor.eval()

    item_ids_global = torch.arange(item_offset, item_offset + num_items, device=device, dtype=torch.long)
    k = int(k)

    window: deque = deque(maxlen=(window_sids if window_sids > 0 else None))
    prefix_sid = int(start_prefix_sid)

    ndcg_big_by_user: Dict[int, List[float]] = {}
    ndcg_small_by_user: Dict[int, List[float]] = {}
    topk_union_big: set = set()
    topk_union_small: set = set()

    for sid in sorted(events_by_sid.keys()):
        while prefix_sid < sid:
            s, d = mp_edges_for_sid(prefix_sid, mp_by_sid, device)
            if s.numel() > 0:
                window.append((s, d))
            prefix_sid += 1

        z_big, z_small = compute_z_from_prefix(window, num_nodes)

        users_global_cpu, pos_items_local_cpu = select_last_event_per_user(events_by_sid[sid], item_offset)
        if users_global_cpu.numel() == 0:
            continue

        users_global = users_global_cpu.to(device)
        pos_items_local = pos_items_local_cpu.to(device)

        users_z_big = z_big[users_global]
        items_z_big = z_big[item_ids_global]
        logits_big = users_z_big @ items_z_big.t()
        topk_big = torch.topk(logits_big, k=k, dim=1).indices

        users_z_small = z_small[users_global]
        items_z_small = z_small[item_ids_global]
        logits_small = users_z_small @ items_z_small.t()
        topk_small = torch.topk(logits_small, k=k, dim=1).indices

        for row in range(int(users_global.numel())):
            u = int(users_global[row].item())
            pos = int(pos_items_local[row].item())

            for topk, ndcg_dict, union_set in [
                (topk_big, ndcg_big_by_user, topk_union_big),
                (topk_small, ndcg_small_by_user, topk_union_small),
            ]:
                hits = (topk[row] == pos).nonzero(as_tuple=False)
                if hits.numel() == 0:
                    ndcg = 0.0
                else:
                    rank0 = int(hits[0].item())
                    ndcg = 1.0 / float(np.log2(rank0 + 2.0))

                ndcg_dict.setdefault(u, []).append(ndcg)
                union_set.update([int(x) for x in topk[row].detach().cpu().tolist()])

    def mean_of_user_means(d: Dict[int, List[float]]) -> float:
        if len(d) == 0:
            return 0.0
        return float(np.mean([float(np.mean(v)) for v in d.values()]))

    ndcg_big = mean_of_user_means(ndcg_big_by_user)
    ndcg_small = mean_of_user_means(ndcg_small_by_user)
    coverage_big = float(len(topk_union_big)) / float(max(1, num_items))
    coverage_small = float(len(topk_union_small)) / float(max(1, num_items))

    return {
        "ndcg_big": ndcg_big,
        "ndcg_small": ndcg_small,
        "coverage_big": coverage_big,
        "coverage_small": coverage_small,
    }

Визуализация промежуточных градиентов

In [13]:
def log_gradient_stats():
    grad_stats = {}

    for name, p in encoder.named_parameters():
        if p.grad is not None:
            grad_stats[f"grad/encoder/{name}_mean"] = p.grad.abs().mean().item()
            grad_stats[f"grad/encoder/{name}_max"] = p.grad.abs().max().item()

    for name, p in compressor.named_parameters():
        if p.grad is not None:
            grad_stats[f"grad/compressor/{name}_mean"] = p.grad.abs().mean().item()
            grad_stats[f"grad/compressor/{name}_max"] = p.grad.abs().max().item()

    for name, p in node_emb.named_parameters():
        if p.grad is not None:
            grad_stats[f"grad/node_emb/{name}_mean"] = p.grad.abs().mean().item()

    wandb.log(grad_stats)


@torch.no_grad()
def visualize_predictions(events_by_sid, title="predictions"):
    encoder.eval()
    compressor.eval()

    item_ids_global = torch.arange(item_offset, item_offset + num_items, device=device)

    sid = sorted(events_by_sid.keys())[0]
    batch = events_by_sid[sid]

    users, pos_items = select_last_event_per_user(batch, item_offset)
    if users.numel() == 0:
        return

    src = torch.tensor(batch["from"].values, device=device)
    dst = torch.tensor(batch["to"].values, device=device)

    A = build_norm_adj(src, dst, num_nodes, device)
    z_big = encoder(A, node_emb.weight)
    z_small = compressor(z_big)

    logits_big = z_big[users.to(device)] @ z_big[item_ids_global].t()
    logits_small = z_small[users.to(device)] @ z_small[item_ids_global].t()

    top_big = torch.topk(logits_big, 10, dim=1).indices.cpu().numpy()
    top_small = torch.topk(logits_small, 10, dim=1).indices.cpu().numpy()

    table = wandb.Table(columns=["user","gt","top_big","top_small"])

    for i in range(min(20, len(users))):
        table.add_data(
            int(users[i]),
            int(pos_items[i]),
            top_big[i].tolist(),
            top_small[i].tolist()
        )

    wandb.log({f"predictions/{title}": table})

Training runner

In [14]:
def init_models_and_opt(cfg, device):
    node_emb = nn.Embedding(num_nodes, cfg.node_dim).to(device)
    torch.nn.init.normal_(node_emb.weight, std=0.1)

    encoder = SimpleGCNEncoder(
        in_dim=cfg.node_dim,
        hid_dim=cfg.embed_dim,
        out_dim=cfg.embed_dim,
        n_layers=cfg.n_layers,
        dropout=cfg.dropout,
    ).to(device)

    compressor = Compressor(cfg.embed_dim, cfg.compressed_dim).to(device)

    opt = torch.optim.Adam(
        list(node_emb.parameters()) + list(encoder.parameters()) + list(compressor.parameters()),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )
    return node_emb, encoder, compressor, opt


def init_scheduler(cfg, opt):
    # только на фулл
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=int(cfg.full_train_epochs),
    )


cfg_full = CFG(**vars(cfg))  # копия текущего конфига под full-run

In [15]:
# FULL: init под обычные настройки
node_emb, encoder, compressor, opt = init_models_and_opt(cfg_full, device)
scheduler = init_scheduler(cfg_full, opt)

wandb.init(project="embedding_compression", name="full_dataset", config=cfg_full.__dict__)
print("\nFull-dataset learning")

# метрики до обучения
val0 = eval_streaming(val_events_by_sid, mp_by_sid, 0, cfg_full.window_sids, cfg_full.k)
wandb.log({"epoch": 0, "lr": float(opt.param_groups[0]["lr"]),
           "val_ndcg_big": val0["ndcg_big"], "val_cov_big": val0["coverage_big"],
           "val_ndcg_small": val0["ndcg_small"], "val_cov_small": val0["coverage_small"]})

best_val = -1.0
best_state = None

for epoch in range(1, cfg_full.full_train_epochs + 1):
    loss = train_epoch_streaming(train_events_by_sid, mp_by_sid, cfg_full.window_sids, debug=False)

    # шаг scheduler 1 раз на эпоху
    scheduler.step()

    train_metrics = eval_streaming(
        events_by_sid=train_events_by_sid,
        mp_by_sid=mp_by_sid,
        start_prefix_sid=0,
        window_sids=cfg_full.window_sids,
        k=cfg_full.k,
    )

    val = eval_streaming(
        events_by_sid=val_events_by_sid,
        mp_by_sid=mp_by_sid,
        start_prefix_sid=0,
        window_sids=cfg_full.window_sids,
        k=cfg_full.k,
    )

    wandb.log({
        "epoch": epoch,
        "lr": float(opt.param_groups[0]["lr"]),
        "train_loss": loss,

        "train_ndcg_big": train_metrics["ndcg_big"],
        "train_ndcg_small": train_metrics["ndcg_small"],
        "train_cov_big": train_metrics["coverage_big"],
        "train_cov_small": train_metrics["coverage_small"],

        "val_ndcg_big": val["ndcg_big"],
        "val_ndcg_small": val["ndcg_small"],
        "val_cov_big": val["coverage_big"],
        "val_cov_small": val["coverage_small"],
    })

    print(
        f"\nEpoch {epoch:03d} | lr={opt.param_groups[0]['lr']:.2e} | loss={loss:.4f}\n"
        f"val:  NDCG_big@{cfg_full.k}={val['ndcg_big']:.4f}   "
        f"NDCG_small@{cfg_full.k}={val['ndcg_small']:.4f}   "
        f"Cov_big@{cfg_full.k}={val['coverage_big']:.4f}   "
        f"Cov_small@{cfg_full.k}={val['coverage_small']:.4f}"
    )

    if val["ndcg_big"] > best_val:
        best_val = float(val["ndcg_big"])
        best_state = {
            "epoch": epoch,
            "node_emb": {k: v.detach().cpu().clone() for k, v in node_emb.state_dict().items()},
            "encoder":  {k: v.detach().cpu().clone() for k, v in encoder.state_dict().items()},
            "compressor": {k: v.detach().cpu().clone() for k, v in compressor.state_dict().items()},
        }

print(f"Best val NDCG big: {best_val:.4f}")
wandb.finish()


Full-dataset learning

Epoch 001 | lr=3.85e-05 | loss=23.1026
val:  NDCG_big@20=0.0091   NDCG_small@20=0.0076   Cov_big@20=0.0999   Cov_small@20=0.3639

Epoch 002 | lr=1.66e-04 | loss=18.0591
val:  NDCG_big@20=0.0091   NDCG_small@20=0.0187   Cov_big@20=0.1011   Cov_small@20=0.3371

Epoch 003 | lr=2.38e-04 | loss=16.3603
val:  NDCG_big@20=0.0102   NDCG_small@20=0.0239   Cov_big@20=0.1118   Cov_small@20=0.3442

Epoch 004 | lr=3.28e-06 | loss=15.6559
val:  NDCG_big@20=0.0112   NDCG_small@20=0.0251   Cov_big@20=0.1088   Cov_small@20=0.3002

Epoch 005 | lr=2.80e-04 | loss=15.0690
val:  NDCG_big@20=0.0127   NDCG_small@20=0.0227   Cov_big@20=0.1112   Cov_small@20=0.2836

Epoch 006 | lr=1.04e-04 | loss=14.7505
val:  NDCG_big@20=0.0167   NDCG_small@20=0.0294   Cov_big@20=0.1171   Cov_small@20=0.2990

Epoch 007 | lr=8.90e-05 | loss=14.4547
val:  NDCG_big@20=0.0178   NDCG_small@20=0.0287   Cov_big@20=0.1118   Cov_small@20=0.2794

Epoch 008 | lr=2.87e-04 | loss=14.2319
val:  NDCG_big@20=0.0217   

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇███
grad/compressor/net.0.bias_max,▁▁▄▅▆▆▆▇▅▆▅▅▅█▃▄▅▂▄▃▃▄▅▃▄▄▃▄▄▃
grad/compressor/net.0.bias_mean,▁▁▅█▇███▆▆▆▆▇▇▄▅▅▃▅▄▄▄▅▃▄▅▄▄▄▃
grad/compressor/net.0.weight_max,▃▁▂▄▃▂▃▅▅▄▄▆▅▄▅▇▃▇▅▄▄▆▅▄▅█▆▇▇▆
grad/compressor/net.0.weight_mean,▃▃▁▂▂▁▂▄▄▃▃▆▅▄▅▆▅▆▇▅▆▇█▅▇▇▇█▇█
grad/compressor/net.2.bias_max,▃▁▇▇▆█▆▅▆█▆▆▇█▆▄▆▄▆▆▃▅▄▃▅▅▅▄▄▃
grad/compressor/net.2.bias_mean,▁▁▇███▇▆▆▆▆▅▅▅▄▅▃▃▃▄▃▂▃▂▃▃▃▂▃▁
grad/compressor/net.2.weight_max,▄▁▅▄▄▇█▄▆▅▆█▅▆▅▆▆▅▅▅▅▆▆▃▅▆▆▆█▇
grad/compressor/net.2.weight_mean,▂▁▅▆▆▆▅▄▇▆▆█▆▅▅█▅▅▅▅▆▆▅▄▅▆▅▅▅▆
grad/compressor/net.3.bias_max,█▂▃▃▂▂▂▂▂▂▃▃▂▂▂▂▂▁▂▂▁▂▂▁▁▁▂▂▂▁
+26,...


In [20]:
# SANITY: отдельный конфиг + re-init
cfg_sanity = CFG(**vars(cfg_full))
cfg_sanity.dropout = 0.0
cfg_sanity.weight_decay = 0.0
cfg_sanity.distillation_weight = 0.0
cfg_sanity.grad_clip = 0.0

node_emb, encoder, compressor, opt = init_models_and_opt(cfg_sanity, device)
scheduler = None  # sanity: не используем

wandb.init(project="embedding_compression", name="debug_dataset", config=cfg_sanity.__dict__)
print("Overfitting on debug-dataset")

debug_train = {
    sid: train_events_by_sid[sid]
    for sid in sorted(train_events_by_sid.keys())[: cfg_sanity.debug_sids]
}

# метрики до обучения
val0 = eval_streaming(debug_train, mp_by_sid, 0, cfg_sanity.window_sids, cfg_sanity.k)
wandb.log({"epoch": 0,
           "train_ndcg_big": val0["ndcg_big"], "train_cov_big": val0["coverage_big"],
           "train_ndcg_small": val0["ndcg_small"], "train_cov_small": val0["coverage_small"]})

for epoch in range(1, cfg_sanity.epochs + 1):
    loss = train_epoch_streaming(debug_train, mp_by_sid, cfg_sanity.window_sids, debug=True)

    train_metrics = eval_streaming(
        events_by_sid=debug_train,
        mp_by_sid=mp_by_sid,
        start_prefix_sid=0,
        window_sids=cfg_sanity.window_sids,
        k=cfg_sanity.k,
    )

    wandb.log({
        "epoch": epoch,
        "train_loss": loss,
        "train_ndcg_big": train_metrics["ndcg_big"],
        "train_ndcg_small": train_metrics["ndcg_small"],
        "train_cov_big": train_metrics["coverage_big"],
        "train_cov_small": train_metrics["coverage_small"],
    })

    if epoch == 1 or epoch % 10 == 0:
        print(
            f"\nEpoch {epoch} | loss={loss:.4f}\n"
            f"train: NDCG_big@{cfg_sanity.k}={train_metrics['ndcg_big']:.4f}   "
            f"NDCG_small@{cfg_sanity.k}={train_metrics['ndcg_small']:.4f}   "
            f"Cov_big@{cfg_sanity.k}={train_metrics['coverage_big']:.4f}   "
            f"Cov_small@{cfg_sanity.k}={train_metrics['coverage_small']:.4f}"
        )

wandb.finish()

epoch,▁
train_cov_big,▁
train_cov_small,▁
train_ndcg_big,▁
train_ndcg_small,▁
epoch,0
train_cov_big,0.04281
train_cov_small,0.25981
train_ndcg_big,0
train_ndcg_small,0.01573


Overfitting on debug-dataset

Epoch 1 | loss=30.0684
train: NDCG_big@20=0.0000   NDCG_small@20=0.0000   Cov_big@20=0.0731   Cov_small@20=0.2705

Epoch 10 | loss=15.7240
train: NDCG_big@20=0.1429   NDCG_small@20=0.1075   Cov_big@20=0.1897   Cov_small@20=0.2658

Epoch 20 | loss=8.2961
train: NDCG_big@20=0.4485   NDCG_small@20=0.6935   Cov_big@20=0.1831   Cov_small@20=0.2788

Epoch 30 | loss=5.0037
train: NDCG_big@20=0.5722   NDCG_small@20=0.9749   Cov_big@20=0.1825   Cov_small@20=0.2503

Epoch 40 | loss=3.8657
train: NDCG_big@20=0.6368   NDCG_small@20=1.0000   Cov_big@20=0.1742   Cov_small@20=0.2372

Epoch 50 | loss=3.4201
train: NDCG_big@20=0.6383   NDCG_small@20=1.0000   Cov_big@20=0.1718   Cov_small@20=0.2140


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
grad/compressor/net.0.bias_max,▁▂▂▃▄▃▃▄▄▄▃▃▄▅▅▅▅▅▆▇███▇▇▇▆▅▇▆▇▃▇▆▅▄▅▆▇█
grad/compressor/net.0.bias_mean,▁▂▂▃▄▅▆▆▆▆▄▆▆▆▅▅▆▆▆▆▇██▇█▇▇▆▇▇▆█▆▇▇▇▆▆▇▅
grad/compressor/net.0.weight_max,▁▁▁▂▂▂▂▂▂▃▄▄▄▃▃▄▄▄▄▄▄▃▃▃▄▄▄▅▃▄▄▅▃▃▄▄▆▄▇█
grad/compressor/net.0.weight_mean,▁▂▂▃▄▅▅▅▆▆▇▇▇▇▇▆▆▆▆▆▇▇▇▇▇▇▆▆▆▇▇▇▇██▇█▇▇▇
grad/compressor/net.2.bias_max,▁▃▃▄▄▅▅▆▅▄▄▅▅▆▇██▇▇▆▅▅▅▅▆▆▅▅▅▄▄▅▆▆▅▇▅▇▆▅
grad/compressor/net.2.bias_mean,▁▂▃▄▅▅▅▅▅▅▃▂▃▄▄▆▆▅▅▄▅▅▅▅▅▆▅▇██▆▇▇█▇█▇█▆▅
grad/compressor/net.2.weight_max,▁▁▁▂▂▂▂▃▄▄▄▄▄▄▅▇███▇▇▆▆▅▅▆▆▆▆▆▇█▇▆▇▆▇█▆▅
grad/compressor/net.2.weight_mean,▁▂▃▃▄▄▄▅▅▅▅▅▅▅▆▆▅▅▅▅▆▅▅▅▅▆▆▇▆▇▇▆▇▇██▇█▇▇
grad/compressor/net.3.bias_max,▆▇▇███▇▇▇▆▃▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▂▂▁▂▂▂▂▂▂▂▂
+21,...
